# प्रॉम्प्ट इंजिनीअरिंग परिचय
प्रॉम्प्ट इंजिनीअरिंग म्हणजे नैसर्गिक भाषा प्रक्रिया कार्यांसाठी प्रॉम्प्ट डिझाइन करणे आणि ऑप्टिमाइझ करणे होय. यात योग्य प्रॉम्प्ट निवडणे, त्याच्या पॅरामीटर्सची ट्यूनिंग करणे आणि त्यांच्या कार्यक्षमतेचे मूल्यमापन करणे यांचा समावेश होतो. प्रॉम्प्ट इंजिनीअरिंग NLP मॉडेल्समध्ये उच्च अचूकता आणि कार्यक्षमता साध्य करण्यासाठी अत्यंत महत्त्वाचे आहे. या विभागात, आपण OpenAI मॉडेल्स वापरून प्रॉम्प्ट इंजिनीअरिंगच्या मूलभूत गोष्टींचा अभ्यास करणार आहोत.


### व्यायाम 1: टोकनायझेशन
tiktoken वापरून टोकनायझेशन एक्सप्लोर करा, जो OpenAI कडून एक मुक्त स्रोत जलद टोकनायझर आहे
अधिक उदाहरणांसाठी [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) पहा.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### व्यायाम 2: माइक्रोसॉफ्ट फाउंड्री मॉडेल्स की सेटअपची पडताळणी करा

> **टीप:** GitHub Models जुलै 2026 च्या शेवटी बंद होत आहे. या व्यायामासाठी [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) वापरले आहे, जे तसाच मोफत वापरासाठी उपलब्ध असलेला मॉडेल कॅटलॉग आणि Azure AI Inference SDK अनुभव प्रदान करते.

तुमचा Microsoft Foundry Models endpoint योग्यरित्या सेटअप झाला आहे का हे तपासण्यासाठी खालील कोड चालवा. कोड फक्त एक सोपा बेसिक प्रॉम्प्ट वापरतो आणि पूर्ण होणारी प्रक्रिया पडताळतो. इनपुट `oh say can you see` चा पूर्ण होणारा भाग साधारणपणे `by the dawn's early light..` असा असावा.


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

model_name = "gpt-4o-mini"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### व्यायाम ३: बनावट गोष्टी
जेव्हा तुम्ही LLM ला अशा विषयावर प्रॉम्प्ट देऊन पूर्णता परत मागता जे अस्तित्वात नसेल किंवा ज्याबद्दल त्याला माहिती नसेल कारण ते त्याच्या पूर्व-प्रशिक्षित डेटासेटच्या बाहेरचे (अधिक अलीकडील) असू शकते तेव्हा काय होते ते तपासा. वेगळा प्रॉम्प्ट वापरल्यास किंवा वेगळा मॉडेल वापरल्यास प्रतिसाद कसा बदलतो ते पाहा.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### सराव ४: सूचना आधारित 
प्राइमरी कंटेंट सेट करण्यासाठी "text" व्हेरीएबल वापरा 
आणि त्या प्राइमरी कंटेंटशी संबंधित सूचना प्रदान करण्यासाठी "prompt" व्हेरीएबल वापरा.

येथे आम्ही मॉडेलला दुसरी इयत्तेच्या विद्यार्थ्यासाठी मजकूराचा सारांश बनवायला सांगतो


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### व्यायाम 5: जटिल प्रॉम्प्ट
सिस्टम, वापरकर्ता आणि सहाय्यक संदेश असलेली विनंती करून पहा
सिस्टम सहाय्यकाचा संदर्भ सेट करते
वापरकर्ता आणि सहाय्यक संदेश बहुपट संभाषण संदर्भ प्रदान करतात

लक्षात घ्या की सिस्टम संदर्भात सहाय्यक व्यक्तिमत्त्व "उपहासात्मक" सेट केले आहे.
वेगळे व्यक्तिमत्त्व संदर्भ वापरून पहा. किंवा इनपुट/आउटपुट संदेशांची वेगळी साखळी वापरून पहा


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### व्यायाम: तुमची अंतर्ज्ञान तपासा
वरील उदाहरणे तुम्हाला नवीन प्रॉम्प्ट्स तयार करण्यासाठी नमुने देतात (सोपे, गुंतागुंतीचे, सूचना इत्यादी) - आम्ही चर्चिलेले काही अन्य कल्पना जसे की उदाहरणे, संकेत आणि बरेच काही तपासण्यासाठी इतर व्यायाम तयार करण्याचा प्रयत्न करा.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
